# Predicting Dating App Matches: Feature Engineering, a Baseline Model, and a Two-Tower Neural Network

This notebook walks through building a **match prediction model** for a dating app, end to end:

1. Generate a synthetic dataset of users and swipe behavior (stand-in for real production data)
2. Engineer pairwise features that describe the *relationship* between two people
3. Train a baseline **XGBoost** classifier on those hand-crafted features
4. Build a **two-tower neural network from scratch in NumPy** that learns user embeddings directly from raw profile features, and compare it to the baseline
5. Discuss why each approach wins/loses, and how this would evolve in a real production system

> The two-tower model is implemented with explicit forward/backward passes in NumPy (no PyTorch/TensorFlow required), so this notebook runs anywhere with just `numpy`, `pandas`, `scikit-learn`, and `xgboost`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

## 1. Generate synthetic users

Each user has:
- Demographics: age, gender
- Location: lat/lon (in a normalized -1..1 box, standing in for real coordinates)
- Stated preferences: acceptable age range, preferred gender, max distance
- A 12-dimensional binary "interests" vector (tags they've selected)
- An `activity` score standing in for how generally selective/active a swiper they are

In a real system, these would come from your user & preferences tables, plus derived features like embeddings of bio text or photos.

In [ ]:
N_USERS = 3000
N_INTERESTS = 12

def generate_users(n):
    age = rng.integers(18, 55, size=n)
    gender = rng.integers(0, 2, size=n)          # simplified binary, for demo purposes
    pref_gender = 1 - gender                      # assume hetero preference to keep the synthetic signal simple
    lat = rng.uniform(-1, 1, size=n)
    lon = rng.uniform(-1, 1, size=n)
    interests = rng.integers(0, 2, size=(n, N_INTERESTS))
    pref_age_min = np.clip(age - rng.integers(3, 10, size=n), 18, None)
    pref_age_max = age + rng.integers(3, 10, size=n)
    pref_max_dist = rng.uniform(0.1, 1.5, size=n)
    activity = rng.uniform(0, 1, size=n)
    users = pd.DataFrame({
        "user_id": np.arange(n),
        "age": age, "gender": gender, "pref_gender": pref_gender,
        "lat": lat, "lon": lon,
        "pref_age_min": pref_age_min, "pref_age_max": pref_age_max,
        "pref_max_dist": pref_max_dist, "activity": activity,
    })
    return users, interests

users, interests = generate_users(N_USERS)
users.head()

## 2. Simulate swipes and matches

We sample random candidate pairs and assign each person a probability of swiping right on the other, driven by:
- Whether the candidate falls inside their gender/age/distance preferences
- Interest overlap
- Their general activity/selectiveness

A **match** only happens when *both* people swipe right on each other — this asymmetry (A→B and B→A are separate events) is one of the more important modeling details in this domain.

In [ ]:
N_PAIRS = 40000
idx_a = rng.integers(0, N_USERS, size=N_PAIRS)
idx_b = rng.integers(0, N_USERS, size=N_PAIRS)
mask = idx_a != idx_b
idx_a, idx_b = idx_a[mask], idx_b[mask]

a = users.iloc[idx_a].reset_index(drop=True)
b = users.iloc[idx_b].reset_index(drop=True)
int_a = interests[idx_a]
int_b = interests[idx_b]

def swipe_right_prob(a, b, int_a, int_b):
    dist = np.sqrt((a.lat.values - b.lat.values) ** 2 + (a.lon.values - b.lon.values) ** 2)
    in_age_range = ((b.age.values >= a.pref_age_min.values) & (b.age.values <= a.pref_age_max.values)).astype(float)
    in_dist_range = (dist <= a.pref_max_dist.values).astype(float)
    gender_ok = (b.gender.values == a.pref_gender.values).astype(float)
    interest_overlap = (int_a * int_b).sum(axis=1) / N_INTERESTS
    logit = (
        -4.5
        + 3.0 * gender_ok
        + 1.5 * in_age_range
        + 1.5 * in_dist_range
        + 2.5 * interest_overlap
        + 1.0 * a.activity.values
        + rng.normal(0, 0.4, size=len(a))
    )
    return 1 / (1 + np.exp(-logit))

p_a_swipes_b = swipe_right_prob(a, b, int_a, int_b)
p_b_swipes_a = swipe_right_prob(b, a, int_b, int_a)
a_swipes_b = rng.uniform(size=len(a)) < p_a_swipes_b
b_swipes_a = rng.uniform(size=len(a)) < p_b_swipes_a
match = (a_swipes_b & b_swipes_a).astype(int)

print(f"Generated {len(match):,} candidate pairs")
print(f"Match rate: {match.mean():.1%}")

## 3. Baseline model: hand-crafted pairwise features + XGBoost

This is where most of the modeling value usually comes from in practice: turning "two user profiles" into features that describe their *compatibility*, not just their individual attributes.

In [ ]:
def build_pairwise_features(a, b, int_a, int_b):
    dist = np.sqrt((a.lat.values - b.lat.values) ** 2 + (a.lon.values - b.lon.values) ** 2)
    feat = pd.DataFrame({
        "distance": dist,
        "age_diff": np.abs(a.age.values - b.age.values),
        "a_in_b_age_range": ((a.age.values >= b.pref_age_min.values) & (a.age.values <= b.pref_age_max.values)).astype(int),
        "b_in_a_age_range": ((b.age.values >= a.pref_age_min.values) & (b.age.values <= a.pref_age_max.values)).astype(int),
        "a_within_b_dist": (dist <= b.pref_max_dist.values).astype(int),
        "b_within_a_dist": (dist <= a.pref_max_dist.values).astype(int),
        "gender_match_a_wants": (b.gender.values == a.pref_gender.values).astype(int),
        "gender_match_b_wants": (a.gender.values == b.pref_gender.values).astype(int),
        "interest_overlap": (int_a * int_b).sum(axis=1) / N_INTERESTS,
        "a_activity": a.activity.values,
        "b_activity": b.activity.values,
    })
    return feat

X_feat = build_pairwise_features(a, b, int_a, int_b)
y = match

X_train, X_test, y_train, y_test = train_test_split(
    X_feat, y, test_size=0.2, random_state=0, stratify=y
)

baseline_clf = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, eval_metric="auc")
baseline_clf.fit(X_train, y_train)

p_baseline = baseline_clf.predict_proba(X_test)[:, 1]
baseline_auc = roc_auc_score(y_test, p_baseline)
print(f"XGBoost baseline AUC: {baseline_auc:.4f}")

pd.Series(baseline_clf.feature_importances_, index=X_feat.columns).sort_values(ascending=False)

As expected, whether the candidate is even in the preferred gender/age/distance range dominates — these hard compatibility gates carry most of the signal, with interest overlap adding a smaller boost on top.

## 4. Two-tower neural network (from scratch, in NumPy)

The idea behind a **two-tower model**: instead of hand-crafting comparison features, learn an embedding function that maps *any* user's raw profile into a shared vector space, such that compatible people end up with embeddings that have a **high dot product**.

```
user A features ─▶ [ shared tower ] ─▶ embedding_A ┐
                                                      ├─▶ dot product ─▶ sigmoid ─▶ P(match)
user B features ─▶ [ shared tower ] ─▶ embedding_B ┘
```

A few design choices worth calling out:
- **Shared weights.** Because "does A match B" is symmetric with "does B match A," we use *one* tower (one set of weights) to encode both people, rather than a separate "query tower" and "candidate tower" like you'd see in search/recsys (where the two sides are genuinely different things, e.g. a query vs. a document).
- **Dot-product combiner.** This is what makes two-tower models attractive at scale: once trained, you can pre-compute every user's embedding once, then find compatible candidates for a given user via approximate nearest-neighbor search (e.g. FAISS) instead of scoring every pair with a full model — this is the whole point of the architecture in production recommender systems.
- **No PyTorch/TensorFlow.** To keep this notebook dependency-free, the tower is a 2-layer MLP (Linear → ReLU → Linear) with **manually derived gradients** and a from-scratch Adam optimizer. Swap this for `torch.nn.Module` directly if you want autograd — the math below is exactly what autograd would compute for you.

In [ ]:
class TwoTowerNet:
    """
    Shared-weight two-tower model, implemented from scratch in NumPy.
    Both people are encoded by the SAME tower (the problem is symmetric).
    Match score = sigmoid(dot(embedding_A, embedding_B) + bias).
    """
    def __init__(self, n_features, hidden_dim=24, emb_dim=12, seed=0):
        rng = np.random.default_rng(seed)

        def xavier(fan_in, fan_out):
            limit = np.sqrt(6 / (fan_in + fan_out))
            return rng.uniform(-limit, limit, size=(fan_in, fan_out))

        self.W1 = xavier(n_features, hidden_dim); self.b1 = np.zeros(hidden_dim)
        self.W2 = xavier(hidden_dim, emb_dim);     self.b2 = np.zeros(emb_dim)
        self.bias = 0.0

        self.params = ["W1", "b1", "W2", "b2", "bias"]
        self.m = {p: (np.zeros_like(getattr(self, p)) if p != "bias" else 0.0) for p in self.params}
        self.v = {p: (np.zeros_like(getattr(self, p)) if p != "bias" else 0.0) for p in self.params}
        self.t = 0

    def _encode(self, X):
        """Run one tower: raw features -> embedding."""
        z1 = X @ self.W1 + self.b1
        h1 = np.maximum(z1, 0)          # ReLU
        emb = h1 @ self.W2 + self.b2    # linear embedding head (no activation)
        return z1, h1, emb

    def forward(self, Xa, Xb):
        za1, ha1, emb_a = self._encode(Xa)
        zb1, hb1, emb_b = self._encode(Xb)
        logits = np.sum(emb_a * emb_b, axis=1) + self.bias
        p = 1 / (1 + np.exp(-logits))
        cache = (Xa, Xb, za1, ha1, emb_a, zb1, hb1, emb_b, p)
        return p, cache

    def loss(self, p, y, eps=1e-9):
        return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

    def backward(self, cache, y):
        Xa, Xb, za1, ha1, emb_a, zb1, hb1, emb_b, p = cache
        n = len(y)
        dlogits = (p - y) / n  # dL/dlogits

        d_emb_a = dlogits[:, None] * emb_b
        d_emb_b = dlogits[:, None] * emb_a
        d_bias = np.sum(dlogits)

        def encoder_backward(X, z1, h1, d_emb):
            dW2 = h1.T @ d_emb
            db2 = np.sum(d_emb, axis=0)
            dh1 = d_emb @ self.W2.T
            dz1 = dh1 * (z1 > 0)  # ReLU gradient
            dW1 = X.T @ dz1
            db1 = np.sum(dz1, axis=0)
            return dW1, db1, dW2, db2

        dW1_a, db1_a, dW2_a, db2_a = encoder_backward(Xa, za1, ha1, d_emb_a)
        dW1_b, db1_b, dW2_b, db2_b = encoder_backward(Xb, zb1, hb1, d_emb_b)

        # shared weights -> gradients from both towers accumulate
        return {
            "W1": dW1_a + dW1_b, "b1": db1_a + db1_b,
            "W2": dW2_a + dW2_b, "b2": db2_a + db2_b,
            "bias": d_bias,
        }

    def adam_step(self, grads, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.t += 1
        for p in self.params:
            g = grads[p]
            self.m[p] = beta1 * self.m[p] + (1 - beta1) * g
            self.v[p] = beta2 * self.v[p] + (1 - beta2) * (g * g)
            m_hat = self.m[p] / (1 - beta1 ** self.t)
            v_hat = self.v[p] / (1 - beta2 ** self.t)
            update = lr * m_hat / (np.sqrt(v_hat) + eps)
            setattr(self, p, getattr(self, p) - update)

    def predict_proba(self, Xa, Xb):
        p, _ = self.forward(Xa, Xb)
        return p

    def embed(self, X):
        """Precompute a user's embedding once -> enables fast ANN retrieval in production."""
        _, _, emb = self._encode(X)
        return emb

### Sanity check on synthetic data with a known structure

Before trusting the class on the real problem, let's confirm the forward/backward math is correct by fitting it to a toy dataset where the "compatibility" function is known.

In [ ]:
n, d = 2000, 10
Xa_toy = rng.normal(size=(n, d))
Xb_toy = rng.normal(size=(n, d))
true_w = rng.normal(size=d)
score = (Xa_toy @ true_w) * (Xb_toy @ true_w)
y_toy = (1 / (1 + np.exp(-score)) > 0.5).astype(float)

toy_net = TwoTowerNet(n_features=d, hidden_dim=16, emb_dim=8, seed=0)
for epoch in range(20):
    p, cache = toy_net.forward(Xa_toy, Xb_toy)
    grads = toy_net.backward(cache, y_toy)
    toy_net.adam_step(grads, lr=0.05)
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}  loss={toy_net.loss(p, y_toy):.4f}")

p_final = toy_net.predict_proba(Xa_toy, Xb_toy)
print(f"toy training accuracy: {((p_final > 0.5) == y_toy).mean():.3f}")

Good — the model fits a known bilinear structure almost perfectly, confirming the gradients are implemented correctly.

## 5. Train the two-tower model on raw user features

Now the real test: instead of the hand-crafted pairwise features from Section 3, feed the tower each person's **raw profile features** directly, and let it learn its own notion of compatibility.

In [ ]:
def user_raw_features(u, itr):
    age_n = (u.age.values - 18) / (55 - 18)
    gender = u.gender.values.astype(float)
    pref_gender = u.pref_gender.values.astype(float)
    lat_n = u.lat.values
    lon_n = u.lon.values
    pref_age_min_n = (u.pref_age_min.values - 18) / (55 - 18)
    pref_age_max_n = (u.pref_age_max.values - 18) / (55 - 18)
    pref_max_dist_n = u.pref_max_dist.values / 1.5
    activity = u.activity.values
    return np.column_stack([
        age_n, gender, pref_gender, lat_n, lon_n,
        pref_age_min_n, pref_age_max_n, pref_max_dist_n, activity, itr
    ]).astype(float)

Xa_raw = user_raw_features(a, int_a)
Xb_raw = user_raw_features(b, int_b)
y_arr = match.astype(float)

idx_trainval, idx_test = train_test_split(np.arange(len(match)), test_size=0.2, random_state=0, stratify=match)
idx_train, idx_val = train_test_split(idx_trainval, test_size=0.2, random_state=0, stratify=match[idx_trainval])

print(f"train: {len(idx_train):,}  val: {len(idx_val):,}  test: {len(idx_test):,}")

In [ ]:
net = TwoTowerNet(n_features=Xa_raw.shape[1], hidden_dim=24, emb_dim=12, seed=1)

WEIGHT_DECAY = 1e-3
PATIENCE = 8
best_val_auc, best_state, bad_epochs = 0, None, 0
n_train = len(idx_train)

for epoch in range(80):
    perm = np.random.permutation(n_train)
    for start in range(0, n_train, 512):
        idx = idx_train[perm[start:start + 512]]
        p, cache = net.forward(Xa_raw[idx], Xb_raw[idx])
        grads = net.backward(cache, y_arr[idx])
        grads["W1"] = grads["W1"] + WEIGHT_DECAY * net.W1  # L2 regularization
        grads["W2"] = grads["W2"] + WEIGHT_DECAY * net.W2
        net.adam_step(grads, lr=0.01)

    val_auc = roc_auc_score(y_arr[idx_val], net.predict_proba(Xa_raw[idx_val], Xb_raw[idx_val]))
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = {p: (getattr(net, p).copy() if p != "bias" else net.bias) for p in net.params}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (best val AUC {best_val_auc:.4f})")
            break

# restore best checkpoint
for p, v in best_state.items():
    setattr(net, p, v)

p_test = net.predict_proba(Xa_raw[idx_test], Xb_raw[idx_test])
two_tower_auc = roc_auc_score(y_arr[idx_test], p_test)
print(f"Two-tower test AUC: {two_tower_auc:.4f}")

## 6. Compare the two approaches

In [ ]:
comparison = pd.DataFrame({
    "model": ["XGBoost (hand-crafted pairwise features)", "Two-tower NN (raw profile features)"],
    "test_auc": [baseline_auc, two_tower_auc],
})

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(comparison["model"], comparison["test_auc"], color=["#4C72B0", "#DD8452"])
ax.set_ylim(0.5, 1.0)
ax.set_ylabel("Test AUC")
ax.set_title("Baseline vs. Two-Tower Model")
plt.xticks(rotation=15, ha="right")
for i, v in enumerate(comparison["test_auc"]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

comparison

## Discussion: why does XGBoost win here, and does that mean two-tower models are worse?

**On this dataset, no — and the gap is expected, not a bug.** A few things are going on:

1. **The task is dominated by hard threshold/range checks** ("is age within [min, max]?", "is distance under the limit?"). Hand-crafted features turn these into a single clean 0/1 signal that a tree model can split on directly. A dot-product of two learned embeddings *can* represent this kind of containment relationship in theory, but it has to discover that representation itself from raw numbers — which is a harder learning problem, especially with a small hidden layer and ~30K training pairs.
2. **Trees are a strong baseline for structured/tabular data in general.** This isn't unique to dating apps — gradient boosting tends to beat neural nets on small-to-medium tabular datasets, and only pulls ahead when there's a lot more data, or the useful signal is buried in something unstructured (text, images) that networks are better at representing.
3. **Two-tower's real advantage is architectural, not raw accuracy: retrieval at scale.** Once trained, `net.embed(X)` gives you a fixed-size vector per user, computed once and cached. Finding good candidates for a given person becomes an approximate-nearest-neighbor search (e.g. FAISS, ScaNN) over millions of pre-computed embeddings, instead of running a full model on every possible pair. XGBoost's pairwise features can't do this — you have to compute a feature vector and score every candidate pair explicitly, which doesn't scale past a candidate pool of a few thousand.

### How this typically plays out in production
- **Two-stage systems are common**: a two-tower model does cheap large-scale *retrieval* (get candidates down from millions to a few hundred), then a more expensive model (like the XGBoost baseline above, or a full cross-attention network) does *ranking* on that shortlist using richer pairwise features.
- **Two-tower models improve a lot with real embeddings as input** — e.g., feeding in bio-text embeddings (sentence-transformers) or photo embeddings (a vision model) instead of raw tabular features, since that's the regime where learned representations meaningfully outperform hand-crafted ones.
- **More data helps more here than for XGBoost.** Neural approaches generally need more examples to reach their ceiling; with only ~30K pairs, the tower is data-constrained in a way the tree model isn't.

### Next steps you could take this notebook
- Feed engineered "cross features" (age-in-range flags, distance) into the tower's input alongside raw attributes, and see how much of the gap closes.
- Swap the from-scratch NumPy tower for a `torch.nn.Module` version once you have PyTorch available — same architecture, autograd instead of manual gradients.
- Try a genuinely asymmetric two-tower (separate weights for "preferences" vs. "attributes") rather than shared weights, and see if it captures the range-membership pattern more easily.
- Replace synthetic data with real (anonymized/aggregated) swipe logs, and add temporal features (recency, swipe-session position) and social-graph features (mutual connections) discussed earlier.